# Reproduce the "LLMs rewrote Qwen's weights" result in 60 seconds

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/namashworks/weightspec/blob/main/notebooks/quickstart.ipynb)

In November 2026, frontier LLMs were shown to edit another model's weights **directly, with no training**. One edit to **1,023 numbers** took Qwen3.5-0.8B from 8.35% to 37.80% on held-out MATH-500 ([Hu et al.](https://arxiv.org/abs/2611.09382), after [Takehi et al., MIT](https://rikiyat.github.io/blog/weight-space-programming/)).

This notebook uses [**WeightSpec**](https://github.com/namashworks/weightspec), an open format and library for weight edits, to:

1. rebuild that 1,023-value edit from the paper's formula, as a 2.7 KB file;
2. turn MIT's released 1.6 GB edited checkpoint into a 250 KB patch and **check it against the authors' own hashes**;
3. find out what MIT's edit actually does;
4. measure the published results with honest paired statistics.

**CPU only.** WeightSpec reads just the tensors it needs from the Hugging Face Hub, about 25 MB.

In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("weightspec") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "weightspec[tokenizers]"])
import json, urllib.request
import numpy as np
import weightspec as ws
print("weightspec", ws.__version__)

weightspec 0.1.0a1


## 1. Open Qwen3.5-0.8B lazily

`ws.load` reads only the file header. Tensors arrive on demand through HTTP range requests.

In [2]:
qwen = ws.load("Qwen/Qwen3.5-0.8B")
print(qwen)
a = qwen.arch
print(f"{a.num_layers} layers, {a.num_heads} heads of width {a.head_dim}, gated queries: {a.gated_query}")
print("full-attention layers:", [i for i, t in enumerate(a.layer_types) if t == "full_attention"])
print(ws.resolve(qwen, "layer[23].mlp.down.col[507]").describe())

WeightStore(hf://Qwen/Qwen3.5-0.8B@2fc06364715b, 488 tensors, 873,438,784 params, arch=qwen3_5_text)
24 layers, 8 heads of width 256, gated queries: True
full-attention layers: [3, 7, 11, 15, 19, 23]
model.language_model.layers.23.mlp.down_proj.weight[:, 507] (1024, 1)


## 2. The 1,023-value edit, from the paper's formula

Hu et al., Appendix C.1, Eq. 3: take the readout direction of *reconsideration* words minus *forward-progress* words, and subtract it from the residual-stream write direction of MLP unit 507 in layer 23.

In [3]:
doubt    = ws.readout(qwen, ["Wait", "However", "But", "Maybe", "Perhaps"])
progress = ws.readout(qwen, ["Therefore", "Thus", "So", "Now", "The", "First", "Next", "Let"])

patch = ws.Patch.new(qwen, "math-doubt-column", cites=["arXiv:2611.09382"]).add(
    "layer[23].mlp.down.col[507]", (doubt - progress).normalize(), alpha=-1.0)
applied = patch.build(policy=ws.Policy.paper())   # embeddings / head / final norm frozen
print(patch.summary())
assert applied.total_changed == 1023              # exactly the count the paper reports
patch.save("math-doubt-column.json")
print(open("math-doubt-column.json").read()[:900], "...")

Patch 'math-doubt-column'  (wspec 0.1)
  base   hf://Qwen/Qwen3.5-0.8B@2fc06364715b  arch=qwen3_5_text
  op 0   add            layer[23].mlp.down.col[507]  {'alpha': -1.0, 'along': 'auto'}
  result 1,023 stored values changed in 1 tensor(s) (0.0279% of those tensors)
{
  "wspec": "0.1",
  "name": "math-doubt-column",
  "base": {
    "source": "hf://Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "arch": "qwen3_5_text",
    "tensors": {
      "model.language_model.layers.23.mlp.down_proj.weight": "1866748ce87d32cfee34891d2fa393fdfeee77447935f6a31d036e4b80332a48"
    }
  },
  "numerics": {
    "directions": "F64",
    "compute": "F32",
    "store": "native",
    "rounding": "nearest-even"
  },
  "ops": [
    {
      "target": "layer[23].mlp.down.col[507]",
      "op": "add",
      "u": {
        "fn": "normalize",
        "of": {
          "fn": "sum",
          "terms": [
            {
              "fn": "readout",
              "ids": [
            

Anyone can now check the file bit for bit, with no GPU and no trust required:

In [4]:
print(ws.Patch.load("math-doubt-column.json").verify())

OK: rebuilt bit for bit (1,023 values changed)


## 3. MIT's edited checkpoint, as a verifiable patch

MIT released their edited model ([RikiyaT/Qwen3.5-0.8B-weight-space-programming](https://huggingface.co/RikiyaT/Qwen3.5-0.8B-weight-space-programming)) and, in their logs, a manifest with the SHA-256 of every tensor they changed. `ws.diff` turns the checkpoint into a patch. Then we compare our hashes with theirs.

In [5]:
mit = ws.load("RikiyaT/Qwen3.5-0.8B-weight-space-programming")
layers = [f"model.language_model.layers.{L}.mlp.down_proj.weight" for L in (15, 19)]
mit_patch = ws.diff(qwen, mit, tensors=layers, name="takehi2026-exp8")
mit_patch.build(qwen)
mit_patch.save("takehi2026-exp8.wspec")
print(mit_patch.summary().splitlines()[-2])

url = ("https://raw.githubusercontent.com/RikiyaT/lm_understand-logs/main/"
       "exp8_math/runs/p50c/edit/manifest.json")
manifest = json.load(urllib.request.urlopen(url))
for t in manifest["tensors"]:
    ours = mit_patch.result.tensors[t["name"]]
    print(f"layer {t['name'].split('.')[3]}:  base hash matches authors: {mit_patch.base.tensors[t['name']] == t['base_sha256']}"
          f"   edited hash matches authors: {ours.sha256 == t['edited_sha256']}")

  result 130,188 stored values changed in 2 tensor(s) (1.7737% of those tensors)


layer 15:  base hash matches authors: True   edited hash matches authors: True
layer 19:  base hash matches authors: True   edited hash matches authors: True


## 4. What did the MIT edit actually do?

In each layer, exactly 64 MLP units change, and the change has rank one. Fitting it shows it is a single operation: **`project_scale` with k = 6**. The component of those units' write directions along one residual direction û is multiplied by 1 − 6 = −5.

In [6]:
mit_applied = mit_patch.apply()
for name in layers:
    w0 = qwen.load(name).astype(np.float64); w1 = mit_applied.numpy(name).astype(np.float64)
    cols = np.flatnonzero((w0 != w1).any(axis=0))
    d = w1[:, cols] - w0[:, cols]
    U, s, _ = np.linalg.svd(d, full_matrices=False)
    k = -(U[:, 0] @ d) / (U[:, 0] @ w0[:, cols])
    print(f"layer {name.split('.')[3]}: {len(cols)} units changed, rank one (s2/s1 = {s[1]/s[0]:.4f}), "
          f"k = {np.median(k):.3f}")

layer 15: 64 units changed, rank one (s2/s1 = 0.0006), k = 6.000
layer 19: 64 units changed, rank one (s2/s1 = 0.0006), k = 6.000


## 5. Measure the claim honestly

MIT published per-question MATH-500 scores for the base and edited model. `ws.paired` compares them **on the same questions**, clusters samples by question (Hu et al., Eq. 2), and gives a 95% interval. Note how few questions both runs share.

In [7]:
base_url = "https://huggingface.co/RikiyaT/Qwen3.5-0.8B-weight-space-programming/resolve/main/eval/"
for f in ("scores_base_math500.json", "scores_edited_math500.json"):
    urllib.request.urlretrieve(base_url + f, f)
print(ws.paired(ws.load_scores("scores_base_math500.json"), ws.load_scores("scores_edited_math500.json")))

Paired comparison on 25 shared questions (100 base / 100 edited samples)
  base      5.00%
  edited   47.00%
  delta   +42.00 pp   SE 7.60 pp   95% CI [+27.11, +56.89] pp
  sample pairs: 42 wrong->right, 0 right->wrong
  verdict CONFIRMED
  note    0 base-only and 475 edited-only questions were excluded (unpaired)


## 6. Controls: the evidence an edit is specific

Hu et al. accepted their edit only after it beat matched controls. `ws.controls` generates them: a dose sweep, a reversed direction, equal-norm random orthogonal directions and an ablation. Evaluate each one on the same held-out questions as the real edit.

In [8]:
for name, ctrl in ws.controls(patch).items():
    ctrl.build()
    print(f"{name:<10} {ctrl.result.changed:>5} values changed")

dose-0.5    1023 values changed


dose-0.75   1023 values changed


dose-1.5    1023 values changed


dose-2      1023 values changed


reverse     1023 values changed


random-0    1024 values changed


random-1    1024 values changed


random-2    1023 values changed
ablate      1024 values changed


## Next steps

* **Edit your own model:** `ws.load(model)` works with any Hugging Face / PyTorch model. `ws.apply(model, patch)` edits it in place.
* **Let an agent do it:** `pip install "weightspec[mcp]"` then `claude mcp add weightspec -- weightspec mcp --workspace ./wsp`.
* **Share an edit:** `patch.push_to_hub("you/patches")`, then anyone runs `weightspec verify hf://you/patches/<file>`.
* **Read the format:** [RFC-0001](https://github.com/namashworks/weightspec/blob/main/spec/RFC-0001-patch-format.md). Feedback is welcome.

If this was useful, a star on [GitHub](https://github.com/namashworks/weightspec) helps others find it.